# 03 · Building Command-Line Utilities: `sys.argv` & `argparse`

**Module 08 – Modules, Packages & Environments** · Estimated time: 40 min

> Expands 📘 *Creating Command Line Utilities*. Not in the 📗 bootcamp.

### 🎯 What you will learn
- Reading raw arguments with `sys.argv`
- `argparse`: positional args, options, flags, types, defaults, choices, help text
- Sub-commands (like `git add` / `git commit`)
- Exit codes, `stdin`/`stdout`, and making a script installable as a command

## 1. `sys.argv`: the raw arguments

When you run `python greet.py Asha 3`, Python stores the words in `sys.argv`:

In [1]:
%%writefile greet_raw.py
import sys

print("sys.argv =", sys.argv)
if len(sys.argv) < 2:
    print("Usage: python greet_raw.py NAME [TIMES]")
    sys.exit(1)                      # non-zero exit code = error
name = sys.argv[1]
times = int(sys.argv[2]) if len(sys.argv) > 2 else 1
for _ in range(times):
    print(f"Hello, {name}!")

Writing greet_raw.py


In [2]:
import subprocess, sys

def run(*args):
    """Run a script like a terminal would, and show its output and exit code."""
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, f"[exit code {r.returncode}]")

run("greet_raw.py", "Asha", "2")
run("greet_raw.py")

sys.argv = ['greet_raw.py', 'Asha', '2']
Hello, Asha!
Hello, Asha!
 [exit code 0]
sys.argv = ['greet_raw.py']
Usage: python greet_raw.py NAME [TIMES]
 [exit code 1]


Parsing `sys.argv` by hand gets messy fast (optional flags, help text, type conversion, errors). That's what `argparse` is for.

## 2. `argparse` (📘 example, extended)

In [3]:
%%writefile repeat.py
"""Print the contents of a file a number of times."""
import argparse

parser = argparse.ArgumentParser(description="A simple command-line utility.")
parser.add_argument("filename", help="The file to process.")
parser.add_argument("-n", "--number", type=int, default=1,
                    help="Number of times to repeat the output (default: 1).")
parser.add_argument("-u", "--upper", action="store_true", help="Print in UPPERCASE.")
args = parser.parse_args()

try:
    with open(args.filename, encoding="utf-8") as file:
        content = file.read()
except FileNotFoundError:
    parser.error(f"File not found: {args.filename}")     # prints usage + exits with code 2

for _ in range(args.number):
    print(content.upper() if args.upper else content, end="")

Writing repeat.py


In [4]:
with open("my_file.txt", "w", encoding="utf-8") as f:
    f.write("Python CLIs are fun\n")

run("repeat.py", "my_file.txt", "-n", "3")
run("repeat.py", "my_file.txt", "--number", "2", "--upper")

Python CLIs are fun
Python CLIs are fun
Python CLIs are fun
 [exit code 0]
PYTHON CLIS ARE FUN
PYTHON CLIS ARE FUN
 [exit code 0]


`argparse` gives you `--help` for free:

In [5]:
run("repeat.py", "--help")

usage: repeat.py [-h] [-n NUMBER] [-u] filename

A simple command-line utility.

positional arguments:
  filename             The file to process.

options:
  -h, --help           show this help message and exit
  -n, --number NUMBER  Number of times to repeat the output (default: 1).
  -u, --upper          Print in UPPERCASE.
 [exit code 0]


...and validates input for you:

In [6]:
run("repeat.py", "my_file.txt", "-n", "three")
run("repeat.py", "missing.txt")

usage: repeat.py [-h] [-n NUMBER] [-u] filename
repeat.py: error: argument -n/--number: invalid int value: 'three'
 [exit code 2]
usage: repeat.py [-h] [-n NUMBER] [-u] filename
repeat.py: error: File not found: missing.txt
 [exit code 2]


## 3. More `add_argument` options

| Option | Example | Meaning |
|---|---|---|
| positional | `add_argument("src")` | required, by position |
| optional | `add_argument("-o", "--output")` | optional, by name |
| `type=` | `type=int`, `type=float`, `type=pathlib.Path` | convert the string |
| `default=` | `default=10` | value if omitted |
| `required=True` | | make an option mandatory |
| `choices=` | `choices=["json", "csv"]` | restrict allowed values |
| `action="store_true"` | `--verbose` | a boolean flag |
| `action="count"` | `-vvv` | count repetitions |
| `nargs=` | `nargs="+"`, `"*"`, `"?"`, `2` | number of values |
| `metavar`, `help` | | help-text display |

In [7]:
import argparse
parser = argparse.ArgumentParser(prog="convert")
parser.add_argument("files", nargs="+", help="input files")
parser.add_argument("--format", choices=["json", "csv"], default="json")
parser.add_argument("-v", "--verbose", action="count", default=0)
# parse_args() normally reads sys.argv; passing a list is handy for testing:
print(parser.parse_args(["a.txt", "b.txt", "--format", "csv", "-vv"]))

Namespace(files=['a.txt', 'b.txt'], format='csv', verbose=2)


## 4. Sub-commands (like `git commit`, `pip install`)

In [8]:
%%writefile todo.py
import argparse, json, pathlib, sys

DB = pathlib.Path("todos.json")

def load():
    return json.loads(DB.read_text(encoding="utf-8")) if DB.exists() else []

def save(todos):
    DB.write_text(json.dumps(todos, indent=2), encoding="utf-8")

def cmd_add(args):
    todos = load()
    todos.append({"task": " ".join(args.task), "done": False})
    save(todos)
    print(f"Added task #{len(todos)}")

def cmd_list(args):
    for i, t in enumerate(load(), start=1):
        print(f"{i}. [{'x' if t['done'] else ' '}] {t['task']}")

def cmd_done(args):
    todos = load()
    if not 1 <= args.number <= len(todos):
        print("No such task", file=sys.stderr)
        return 1
    todos[args.number - 1]["done"] = True
    save(todos)
    print(f"Completed task #{args.number}")

def main() -> int:
    parser = argparse.ArgumentParser(description="Tiny to-do manager")
    sub = parser.add_subparsers(dest="command", required=True)

    p_add = sub.add_parser("add", help="add a task")
    p_add.add_argument("task", nargs="+")
    p_add.set_defaults(func=cmd_add)

    p_list = sub.add_parser("list", help="list tasks")
    p_list.set_defaults(func=cmd_list)

    p_done = sub.add_parser("done", help="mark a task done")
    p_done.add_argument("number", type=int)
    p_done.set_defaults(func=cmd_done)

    args = parser.parse_args()
    return args.func(args) or 0

if __name__ == "__main__":
    sys.exit(main())

Writing todo.py


In [9]:
run("todo.py", "add", "Learn", "argparse")
run("todo.py", "add", "Write tests")
run("todo.py", "done", "1")
run("todo.py", "list")
run("todo.py", "done", "9")

Added task #1
 [exit code 0]


Added task #2
 [exit code 0]


Completed task #1
 [exit code 0]


1. [x] Learn argparse
2. [ ] Write tests
 [exit code 0]
No such task
 [exit code 1]


## 5. Exit codes, stdin & stdout
- Exit code **0 = success**, anything else = failure (`sys.exit(1)`); shells and CI rely on this.
- Print errors to **`sys.stderr`**, results to **stdout**, so users can pipe output: `python todo.py list > tasks.txt`
- Read piped input from `sys.stdin`:

In [10]:
%%writefile upper.py
import sys
for line in sys.stdin:
    sys.stdout.write(line.upper())

Writing upper.py


In [11]:
r = subprocess.run([sys.executable, "upper.py"], input="hello\nworld\n", capture_output=True, text=True)
print(r.stdout)

HELLO
WORLD



## 6. Making it a real command
In `pyproject.toml` (lesson 03):
```toml
[project.scripts]
todo = "todo:main"
```
After `pip install -e .` (or `uv tool install .`) you can type `todo add "buy milk"` anywhere.

> 💡 Popular third-party CLI libraries: **Typer** (uses type hints) and **Click**. And **Rich** for beautiful terminal output.

## 📝 Summary
- `sys.argv` holds raw arguments; `argparse` parses, converts, validates and documents them.
- Positional vs optional args, `type`, `default`, `choices`, `store_true`, `nargs`.
- Sub-commands via `add_subparsers()`.
- Return proper exit codes; errors to stderr.

## 🏋️ Practice
1. Write `wc.py FILE [--lines] [--words] [--chars]` that prints the requested counts (all three if none given).
2. Write `calc.py` with sub-commands `add` and `mul`, each taking two or more numbers (`nargs="+"`, `type=float`).

<details><summary>Show solutions</summary>

```python
# wc.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("file")
p.add_argument("--lines", action="store_true")
p.add_argument("--words", action="store_true")
p.add_argument("--chars", action="store_true")
a = p.parse_args()
text = open(a.file, encoding="utf-8").read()
show_all = not (a.lines or a.words or a.chars)
if a.lines or show_all: print("lines:", len(text.splitlines()))
if a.words or show_all: print("words:", len(text.split()))
if a.chars or show_all: print("chars:", len(text))

# calc.py
import argparse, math
p = argparse.ArgumentParser()
sub = p.add_subparsers(dest="op", required=True)
for name in ("add", "mul"):
    s = sub.add_parser(name)
    s.add_argument("nums", nargs="+", type=float)
a = p.parse_args()
print(sum(a.nums) if a.op == "add" else math.prod(a.nums))
```
</details>